# Lumos — Flickr8k (End-to-End Notebook)

## Goal

Build and validate a **production-like multimodal retrieval pipeline** based on:

- **OpenCLIP (ViT-B/32)** for embeddings
- **Qdrant** for vector search (cosine similarity)
- **Evaluation harness** for retrieval metrics + latency profiling
- (Later) **FastAPI + Streamlit** for a thin serving/demo layer

This notebook is the **single end-to-end driver** for the project v1:
**data sanity → embeddings → indexing → search → evaluation → latency**.

> Scope note: we do **NOT** fine-tune CLIP. We use OpenCLIP as a strong pretrained backbone and focus on retrieval engineering.

---

## What this notebook will do (v1)

### 0) Environment & Config
- Check GPU / CUDA / torch
- Set global config (model, device, precision, batch sizes, paths)
- Fix random seeds for reproducibility

### 1) Dataset sanity-check (Flickr8k)
- Parse `captions.txt`
- Verify all referenced images exist in `data/flickr8k/images`
- Inspect caption stats (lengths, examples)
- Create deterministic **train/test split** (seeded)

### 2) Embedding extraction (OpenCLIP)
- Compute **image embeddings** in batches (fp16 on GPU)
- Compute **text embeddings** for captions in batches
- Normalize embeddings (cosine-ready)
- Save artifacts:
  - `artifacts/flickr8k/meta.parquet`
  - `artifacts/embeddings/image_emb.npy`
  - `artifacts/embeddings/text_emb.npy`
  - `artifacts/model_info.json`

### 3) Qdrant indexing
- Start/ensure Qdrant is running (project-specific instance)
- Create collections and schema
- Upload points with payload:
  - `image_id`, `filename`, `split`
  - (optional) `captions` (or first caption for UI)

### 4) Retrieval demos
- **Text → Image** search (top-K)
- **Image → Image** similarity (top-K)
- (For evaluation) **Image → Text** search (optional)

### 5) Evaluation harness
Compute retrieval metrics on the test split:
- Recall@1/5/10
- mAP@10
- nDCG@10
- MRR@10

Save:
- `artifacts/eval/results.json`
- `artifacts/eval/results_table.csv` (optional)

### 6) Latency profiling
Measure:
- embedding latency (image/text)
- Qdrant search latency
- end-to-end latency (encode + search)

Save:
- `artifacts/eval/latency.json`

---

## Dataset layout (local)

Root:
- `data/flickr8k/images/*.jpg`  (~8091 files)
- `data/flickr8k/captions.txt`

Artifacts produced by this notebook:
- `artifacts/` (generated)
  - `flickr8k/`
  - `embeddings/`
  - `eval/`

---

## Notes on correctness & reproducibility

We will ensure:
- deterministic split (seed)
- no leakage between train/test for evaluation
- consistent embedding normalization
- clear mapping between:
  - images ↔ captions ↔ embeddings ↔ qdrant points

---

## Next steps after this notebook
Once the pipeline and metrics are stable:
- Implement minimal **FastAPI** endpoints:
  - `/search_text`, `/search_image`, `/health`
- Implement minimal **Streamlit** UI for demo
- Wrap everything into a single **docker-compose**:
  - Qdrant + API + UI

---

## Expected outcome of v1
A reproducible, benchmarked multimodal retrieval system:
- end-to-end pipeline
- saved artifacts
- quantitative metrics
- latency profile
- ready for thin serving layer

In [ ]:
# Imports + paths + reproducibility

import json
import numpy as np
import math
import open_clip
import os
import pandas as pd
import random
import re
import requests
import time
import torch

from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime
from IPython.display import display
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from typing import Dict, Iterable, List, Tuple
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from qdrant_client.http.models import PointStruct

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "flickr8k"
IMAGES_DIR = DATA_DIR / "images"
CAPTIONS_PATH = DATA_DIR / "captions.txt"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("CAPTIONS_PATH exists:", CAPTIONS_PATH.exists())

In [ ]:
# Torch / CUDA check

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("cuda version:", torch.version.cuda)
    print("gpu:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

In [ ]:
# Config

@dataclass(frozen=True)
class Config:
    seed: int = 42

    # OpenCLIP
    clip_model: str = "ViT-B-32"
    clip_pretrained: str = "openai"     
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_fp16: bool = True 

    # batches
    batch_size_images: int = 128   
    batch_size_texts: int = 256

    # dataset split
    test_size: float = 0.2 
    split_seed: int = 42

CFG = Config()
CFG

In [ ]:
# Reproducibility seeds

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG.seed)
print("Seed set:", CFG.seed)

In [ ]:
# Captions parser

def parse_flickr8k_captions(path: Path) -> pd.DataFrame:
    """
    Returns DataFrame with columns:
      - filename (str)
      - caption (str)
      - cap_id (int | None)
    Supports:
      1) "xxx.jpg\tcaption"
      2) "xxx.jpg#0\tcaption"
    """
    if not path.exists():
        raise FileNotFoundError(f"Captions file not found: {path}")

    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            # try split by tab first (most common)
            if "\t" in line:
                left, caption = line.split("\t", 1)
            else:
                # fallback: split by first comma or whitespace (rare)
                parts = re.split(r"[,\s]+", line, maxsplit=1)
                if len(parts) < 2:
                    continue
                left, caption = parts[0], parts[1]

            left = left.strip()
            caption = caption.strip()

            # handle "file.jpg#0"
            cap_id = None
            if "#" in left:
                fname, tail = left.split("#", 1)
                left = fname
                try:
                    cap_id = int(tail)
                except ValueError:
                    cap_id = None

            # basic guard
            if not left.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                # if captions contain weird header lines
                continue
            if len(caption) == 0:
                continue

            rows.append((left, caption, cap_id))

    df = pd.DataFrame(rows, columns=["filename", "caption", "cap_id"])
    return df

captions_df = parse_flickr8k_captions(CAPTIONS_PATH)
captions_df.head(), captions_df.shape

In [ ]:
# Sanity checks 
# files on disk

image_files = {p.name for p in IMAGES_DIR.glob("*")}
print("Images on disk:", len(image_files))

# referenced in captions
unique_caption_files = set(captions_df["filename"].unique())
print("Unique images referenced in captions:", len(unique_caption_files))

missing = sorted(list(unique_caption_files - image_files))
extra = sorted(list(image_files - unique_caption_files))

print("Missing referenced images:", len(missing))
print("Extra images not referenced:", len(extra))

if len(missing) > 0:
    print("Example missing:", missing[:10])

# captions per image
caps_per_img = captions_df.groupby("filename").size()
caps_per_img.describe()

In [ ]:
# Caption quick stats

captions_df["caption_len"] = captions_df["caption"].str.len()
captions_df["caption_words"] = captions_df["caption"].str.split().apply(len)

print(captions_df[["caption_len", "caption_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# sample captions
captions_df.sample(10, random_state=CFG.seed)[["filename", "caption"]]

In [ ]:
# Build image-level meta + deterministic split

meta = (
    captions_df
    .groupby("filename")["caption"]
    .apply(list)
    .reset_index()
    .rename(columns={"caption": "captions"})
)

meta["image_id"] = np.arange(len(meta), dtype=np.int64)

# deterministic split
rng = np.random.default_rng(CFG.split_seed)
perm = rng.permutation(len(meta))
test_n = int(len(meta) * CFG.test_size)
test_idx = set(perm[:test_n].tolist())

meta["split"] = ["test" if i in test_idx else "train" for i in range(len(meta))]

meta.head(), meta["split"].value_counts()

In [ ]:
# Save meta artifact

meta_dir = ARTIFACTS_DIR / "flickr8k"
meta_dir.mkdir(parents=True, exist_ok=True)

meta_path = meta_dir / "meta.parquet"
meta.to_parquet(meta_path, index=False)

print("Saved:", meta_path)
meta_path

In [ ]:
# Load OpenCLIP model + preprocess

device = torch.device(CFG.device)

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=CFG.clip_model,
    pretrained=CFG.clip_pretrained,
    device=device
)
tokenizer = open_clip.get_tokenizer(CFG.clip_model)

model.eval()

print("Loaded:", CFG.clip_model, "| pretrained:", CFG.clip_pretrained)
print("Device:", device)

In [ ]:
# Utils: batching + normalize

def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / (x.norm(dim=-1, keepdim=True) + eps)

def batch_iter(n: int, batch_size: int) -> Iterable[Tuple[int, int]]:
    for i in range(0, n, batch_size):
        yield i, min(i + batch_size, n)

@torch.inference_mode()
def encode_images(filepaths: List[Path], batch_size: int) -> np.ndarray:
    embs = []
    use_amp = (CFG.use_fp16 and device.type == "cuda")

    for s, e in batch_iter(len(filepaths), batch_size):
        batch_paths = filepaths[s:e]
        imgs = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            imgs.append(preprocess(img))
        x = torch.stack(imgs, dim=0).to(device)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            feats = model.encode_image(x)
            feats = l2_normalize(feats)

        embs.append(feats.detach().cpu().to(torch.float32).numpy())

    return np.concatenate(embs, axis=0)

@torch.inference_mode()
def encode_texts(texts: List[str], batch_size: int) -> np.ndarray:
    embs = []
    use_amp = (CFG.use_fp16 and device.type == "cuda")

    for s, e in batch_iter(len(texts), batch_size):
        batch_text = texts[s:e]
        tokens = tokenizer(batch_text).to(device)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            feats = model.encode_text(tokens)
            feats = l2_normalize(feats)

        embs.append(feats.detach().cpu().to(torch.float32).numpy())

    return np.concatenate(embs, axis=0)

In [ ]:
# Build embedding inputs
# Image-level (one vector per image)

meta_sorted = meta.sort_values("image_id").reset_index(drop=True)
image_paths = [IMAGES_DIR / fn for fn in meta_sorted["filename"].tolist()]

cap_rows = []
for _, row in meta_sorted.iterrows():
    image_id = int(row["image_id"])
    filename = row["filename"]
    for cap in row["captions"]:
        cap_rows.append((image_id, filename, cap))

caps_level = pd.DataFrame(cap_rows, columns=["image_id", "filename", "caption"])
print("Image-level N:", len(image_paths))
print("Caption-level M:", len(caps_level))
caps_level.head()

In [ ]:
# Encode images

t0 = time.perf_counter()
image_emb = encode_images(image_paths, batch_size=CFG.batch_size_images)
t1 = time.perf_counter()

print("image_emb shape:", image_emb.shape)
print("Time (s):", round(t1 - t0, 2), "| per image (ms):", round(1000 * (t1 - t0) / len(image_emb), 3))

# Save
emb_dir = ARTIFACTS_DIR / "embeddings"
emb_dir.mkdir(parents=True, exist_ok=True)

image_emb_path = emb_dir / "image_emb.npy"
np.save(image_emb_path, image_emb)
print("Saved:", image_emb_path)

In [ ]:
# Encode captions (texts)

texts = caps_level["caption"].tolist()

t0 = time.perf_counter()
text_emb = encode_texts(texts, batch_size=CFG.batch_size_texts)
t1 = time.perf_counter()

print("text_emb shape:", text_emb.shape)
print("Time (s):", round(t1 - t0, 2), "| per caption (ms):", round(1000 * (t1 - t0) / len(text_emb), 3))

text_emb_path = emb_dir / "text_emb.npy"
np.save(text_emb_path, text_emb)
print("Saved:", text_emb_path)

# Save caption mapping
cap_map_path = ARTIFACTS_DIR / "flickr8k" / "captions_level.parquet"
caps_level.to_parquet(cap_map_path, index=False)
print("Saved:", cap_map_path)

In [ ]:
# Save model info

model_info = {
    "clip_model": CFG.clip_model,
    "clip_pretrained": CFG.clip_pretrained,
    "embedding_dim": int(image_emb.shape[1]),
    "normalize": True,
    "device": CFG.device,
    "use_fp16": CFG.use_fp16,
    "batch_size_images": CFG.batch_size_images,
    "batch_size_texts": CFG.batch_size_texts,
    "seed": CFG.seed,
    "split_seed": CFG.split_seed,
    "test_size": CFG.test_size,
}

model_info_path = ARTIFACTS_DIR / "model_info.json"
with model_info_path.open("w", encoding="utf-8") as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print("Saved:", model_info_path)
model_info

In [ ]:
# Qdrant: client + create collection

QDRANT_URL = "http://localhost:6334"
COLL_IMAGES = "flickr8k_images"

qdrant = QdrantClient(
    url=QDRANT_URL,
    prefer_grpc=False,
    check_compatibility=False
)

dim = int(image_emb.shape[1])

# recreate collection
if qdrant.collection_exists(COLL_IMAGES):
    qdrant.delete_collection(COLL_IMAGES)

qdrant.create_collection(
    collection_name=COLL_IMAGES,
    vectors_config=qm.VectorParams(size=dim, distance=qm.Distance.COSINE),
)

print("Created collection:", COLL_IMAGES)

In [ ]:
# Upload image vectors + payload

payloads = []
for row in meta_sorted.itertuples(index=False):
    payloads.append({
        "image_id": int(row.image_id),
        "filename": row.filename,
        "split": row.split,
        "caption0": row.captions[0] if isinstance(row.captions, list) and len(row.captions) > 0 else ""
    })

points = [
    PointStruct(id=int(i), vector=image_emb[i].tolist(), payload=payloads[i])
    for i in range(len(meta_sorted))
]

# upload in chunks
B = 512
for s in range(0, len(points), B):
    qdrant.upsert(collection_name=COLL_IMAGES, points=points[s:s+B])

print("Uploaded points:", len(points))
print("Collection count:", qdrant.count(COLL_IMAGES, exact=True).count)

In [ ]:
# Text -> Image search via Qdrant REST

QDRANT_REST = "http://localhost:6334"
COLL_IMAGES = "flickr8k_images"

def qdrant_search_rest(collection: str, vector: np.ndarray, top_k: int = 5):
    url = f"{QDRANT_REST}/collections/{collection}/points/search"
    payload = {
        "vector": vector.astype(float).tolist(),
        "limit": int(top_k),
        "with_payload": True
    }
    r = requests.post(url, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["result"]

def search_text(query: str, top_k: int = 5):
    q_emb = encode_texts([query], batch_size=1)[0]
    return qdrant_search_rest(COLL_IMAGES, q_emb, top_k=top_k)

def show_results(hits, title: str = ""):
    print("\nQuery:", title)
    for i, h in enumerate(hits, start=1):
        fn = h["payload"]["filename"]
        score = h["score"]
        print(f"{i}. score={score:.4f} | {fn}")
        img = Image.open(IMAGES_DIR / fn).convert("RGB")
        display(img)

# demo
q = "a dog running on the grass"
hits = search_text(q, top_k=5)
show_results(hits, title=q)

In [ ]:
# Image -> Image demo via Qdrant REST

def search_similar_images(image_id: int, top_k: int = 6):
    v = image_emb[image_id]  # already normalized
    hits = qdrant_search_rest(COLL_IMAGES, v, top_k=top_k)
    return hits

def show_results_rest(hits, title: str = ""):
    print("\n", title)
    for i, h in enumerate(hits, start=1):
        fn = h["payload"]["filename"]
        score = h["score"]
        print(f"{i}. score={score:.4f} | image_id={h['payload']['image_id']} | {fn}")
        img = Image.open(IMAGES_DIR / fn).convert("RGB")
        display(img)

image_id = 0
hits = search_similar_images(image_id, top_k=6)
show_results_rest(hits, title=f"Similar to image_id={image_id} ({meta_sorted.loc[image_id,'filename']})")

In [ ]:
# Recall@K evaluation (text -> image)

def eval_recall_at_ks(test_caps_df: pd.DataFrame, ks=(1,5,10)) -> dict:
    ks = sorted(list(ks))
    hits_needed = ks[-1]
    correct = {k: 0 for k in ks}
    total = len(test_caps_df)

    for row in tqdm(test_caps_df.itertuples(index=False), total=total):
        query = row.caption
        true_image_id = int(row.image_id)

        q_emb = encode_texts([query], batch_size=1)[0]
        hits = qdrant_search_rest(COLL_IMAGES, q_emb, top_k=hits_needed)

        retrieved_ids = [int(h["payload"]["image_id"]) for h in hits]

        for k in ks:
            if true_image_id in retrieved_ids[:k]:
                correct[k] += 1

    return {f"Recall@{k}": correct[k] / total for k in ks}

# build test captions
test_caps = caps_level.merge(
    meta_sorted[["image_id", "split"]],
    on="image_id",
    how="left"
)
test_caps = test_caps[test_caps["split"] == "test"].reset_index(drop=True)

print("Test captions:", len(test_caps))

recalls = eval_recall_at_ks(test_caps, ks=(1,5,10))
recalls

In [ ]:
# Save evaluation results (Recall@K)

eval_dir = ARTIFACTS_DIR / "eval"
eval_dir.mkdir(parents=True, exist_ok=True)

results_recall = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "dataset": "Flickr8k",
    "task": "text_to_image",
    "metrics": {
        "Recall@1": float(recalls["Recall@1"]),
        "Recall@5": float(recalls["Recall@5"]),
        "Recall@10": float(recalls["Recall@10"]),
    },
    "model": model_info,
    "qdrant": {
        "url": QDRANT_REST,
        "collection": COLL_IMAGES,
        "distance": "cosine",
        "index_points": int(len(meta_sorted)),
    },
    "notes": "Full index (train+test images), eval on test captions.",
}

path = eval_dir / "results_recall.json"
with path.open("w", encoding="utf-8") as f:
    json.dump(results_recall, f, indent=2, ensure_ascii=False)

print("Saved:", path)
results_recall

## Flickr8k Benchmark (ViT-B/32, OpenAI weights)

Text → Image retrieval (cosine, full index)

- Recall@1  = 0.294
- Recall@5  = 0.533
- Recall@10 = 0.631

Hardware: (your GPU)
Precision: fp16 inference (if CUDA available)

In [ ]:
# Latency profiling (text->image)

def percentile_ms(arr_s, p):
    return float(np.percentile(np.array(arr_s) * 1000.0, p))

# sample queries
N_LAT = 300
lat_df = test_caps.sample(N_LAT, random_state=CFG.seed).reset_index(drop=True)

encode_s = []
search_s = []
e2e_s = []

top_k_latency = 10

for row in lat_df.itertuples(index=False):
    q = row.caption

    t0 = time.perf_counter()
    q_emb = encode_texts([q], batch_size=1)[0]
    t1 = time.perf_counter()

    _ = qdrant_search_rest(COLL_IMAGES, q_emb, top_k=top_k_latency)
    t2 = time.perf_counter()

    encode_s.append(t1 - t0)
    search_s.append(t2 - t1)
    e2e_s.append(t2 - t0)

latency = {
    "n_queries": int(N_LAT),
    "top_k": int(top_k_latency),
    "encode_ms": {
        "p50": percentile_ms(encode_s, 50),
        "p95": percentile_ms(encode_s, 95),
        "mean": float(np.mean(encode_s) * 1000.0),
    },
    "search_ms": {
        "p50": percentile_ms(search_s, 50),
        "p95": percentile_ms(search_s, 95),
        "mean": float(np.mean(search_s) * 1000.0),
    },
    "e2e_ms": {
        "p50": percentile_ms(e2e_s, 50),
        "p95": percentile_ms(e2e_s, 95),
        "mean": float(np.mean(e2e_s) * 1000.0),
    },
}

latency

In [ ]:
# Save latency metrics

latency_payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "dataset": "Flickr8k",
    "task": "text_to_image",
    "model": model_info,
    "qdrant": {"url": QDRANT_REST, "collection": COLL_IMAGES},
    "latency": latency
}

path = eval_dir / "latency_text_to_image.json"
with path.open("w", encoding="utf-8") as f:
    json.dump(latency_payload, f, indent=2, ensure_ascii=False)

print("Saved:", path)
path

In [ ]:
# MRR@10 and nDCG@10 on test captions

def eval_mrr_ndcg(test_caps_df: pd.DataFrame, k: int = 10) -> dict:
    rr = []
    ndcg = []

    for row in tqdm(test_caps_df.itertuples(index=False), total=len(test_caps_df)):
        q = row.caption
        true_id = int(row.image_id)

        q_emb = encode_texts([q], batch_size=1)[0]
        hits = qdrant_search_rest(COLL_IMAGES, q_emb, top_k=k)
        retrieved = [int(h["payload"]["image_id"]) for h in hits]

        if true_id in retrieved:
            rank = retrieved.index(true_id) + 1  # 1-based
            rr.append(1.0 / rank)
            ndcg.append(1.0 / math.log2(rank + 1))
        else:
            rr.append(0.0)
            ndcg.append(0.0)

    return {
        f"MRR@{k}": float(np.mean(rr)),
        f"nDCG@{k}": float(np.mean(ndcg)),
        # AP@k (single relevant) == RR
        f"mAP@{k} (single-rel == MRR)": float(np.mean(rr)),
    }

extra_metrics = eval_mrr_ndcg(test_caps, k=10)
extra_metrics

In [ ]:
# Save extra metrics

payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "dataset": "Flickr8k",
    "task": "text_to_image",
    "model": model_info,
    "qdrant": {"url": QDRANT_REST, "collection": COLL_IMAGES},
    "metrics": extra_metrics,
    "notes": "For Flickr8k, each caption has a single relevant image_id."
}

path = eval_dir / "results_rank_metrics.json"
with path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved:", path)
payload

In [ ]:
# Showcase queries

showcase_queries = [
    "a dog running on the grass",
    "a man riding a bicycle",
    "people playing football",
    "a child in a pink dress",
    "a person skiing downhill",
]

for q in showcase_queries:
    hits = qdrant_search_rest(COLL_IMAGES, encode_texts([q], batch_size=1)[0], top_k=5)
    show_results_rest(hits, title=q)

In [ ]:
# Build single summary artifact

summary = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "dataset": "Flickr8k",
    "model": model_info,
    "qdrant": {"url": QDRANT_REST, "collection": COLL_IMAGES},
    "metrics": {
        **results_recall["metrics"],
        **extra_metrics,
    },
    "latency_ms": latency,
    "artifacts": {
        "meta": str((ARTIFACTS_DIR / "flickr8k" / "meta.parquet").resolve()),
        "captions_level": str((ARTIFACTS_DIR / "flickr8k" / "captions_level.parquet").resolve()),
        "image_emb": str((ARTIFACTS_DIR / "embeddings" / "image_emb.npy").resolve()),
        "text_emb": str((ARTIFACTS_DIR / "embeddings" / "text_emb.npy").resolve()),
    }
}

path = eval_dir / "summary.json"
with path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:", path)
summary

# Conclusion (v1)

We built an end-to-end multimodal retrieval pipeline on **Flickr8k** using:

- **OpenCLIP ViT-B/32 (OpenAI weights)** for embeddings (fp16 inference on GPU)
- **Qdrant** for vector search (cosine similarity)
- A lightweight evaluation harness (Recall@K, MRR@K, nDCG@K)
- Latency profiling (encode / search / end-to-end)

## Key results (Text → Image)

**Recall@K (test captions, full image index):**
- Recall@1  = 0.294
- Recall@5  = 0.533
- Recall@10 = 0.631

**Ranking metrics:**
- MRR@10  = 0.397
- nDCG@10 = 0.452

## ⏱ Latency (top-10 search, n=300)

- Encode (p50 / p95): ~ 8.9 ms / 11.2 ms
- Qdrant search (p50 / p95): ~ 24.3 ms / 38.7 ms
- End-to-end (p50 / p95): ~ 33.3 ms / 47.7 ms

## Saved artifacts

- `artifacts/flickr8k/meta.parquet`
- `artifacts/flickr8k/captions_level.parquet`
- `artifacts/embeddings/image_emb.npy`
- `artifacts/embeddings/text_emb.npy`
- `artifacts/model_info.json`
- `artifacts/eval/results_recall.json`
- `artifacts/eval/results_rank_metrics.json`
- `artifacts/eval/latency_text_to_image.json`
- `artifacts/eval/summary.json`

## Next step (v1 serving)

Build a thin serving layer using saved artifacts + Qdrant:

- **FastAPI**
  - `/health`
  - `/search_text`
  - `/search_image`
- **Streamlit UI**
  - Text → Image search
  - Image → Image similarity search

All services will be started via **docker-compose** (Qdrant + API + UI).